# Automatic Context Editing

Long-running agents accumulate bulky tool outputs that fill the context window and cost tokens. `ContextEditingMiddleware` automatically **prunes old tool results** once the conversation crosses a token threshold - keeping the most recent ones and replacing older ones with a small placeholder.

Important: context editing only trims the messages **sent to the model** for that call - it does **not** rewrite your saved conversation history. So we inspect what the model actually receives, not the stored state.

This is the *automatic* counterpart to the manual `trim_messages` / `filter_messages` approach in the **managing messages** notebook.

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## A bulky tool + a low trigger

`fetch_logs` returns a large blob. We configure `ClearToolUsesEdit` with a **low token trigger** so clearing kicks in after a couple of calls, keeping only the most recent tool output.

We also add a small **probe** middleware *after* the context editor. Because middleware nests in list order, the probe is the inner layer and therefore sees the messages **after** editing - exactly what the model receives.

In [2]:
from langchain.agents.middleware import (
    ContextEditingMiddleware,
    ClearToolUsesEdit,
    wrap_model_call,
)
from langgraph.checkpoint.memory import InMemorySaver


@tool
def fetch_logs(service: str) -> str:
    """Fetch recent logs for a service (returns a large blob)."""
    return f"[{service}] " + ("log-line " * 80)


# Inner probe: prints the tool messages the MODEL actually receives,
# i.e. AFTER the context editor has run.
@wrap_model_call
def show_model_inputs(request, handler):
    tool_msgs = [m for m in request.messages if m.type == "tool"]
    if tool_msgs:
        print(f"  -> model receives {len(tool_msgs)} tool message(s):")
        for m in tool_msgs:
            text = m.content if isinstance(m.content, str) else str(m.content)
            print("       ", (text[:45] + " ...") if len(text) > 45 else text)
    return handler(request)


agent = create_agent(
    model=llm_noreason,
    tools=[fetch_logs],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=200,                    # start clearing past ~200 tokens
                    keep=1,                          # keep the most recent tool result
                    placeholder="[old logs cleared]",
                )
            ],
        ),
        show_model_inputs,   # inner layer: sees the edited messages
    ],
    checkpointer=InMemorySaver(),
)


## Run a few turns and watch the model's inputs shrink
We ask the agent to fetch logs for three services on the same thread. By the later turns enough tool output has piled up to cross the trigger, so the probe shows older blobs replaced with `[old logs cleared]` while the newest one is kept in full.

In [4]:
config = make_thread_config()

for i, q in enumerate(
    [
        "Fetch logs for auth-service and give a one-line summary.",
        "Now fetch logs for billing-service and summarize in one line.",
        "Finally fetch logs for search-service and summarize in one line.",
    ],
    start=1,
):
    print(f"=== Turn {i}: {q} ===")
    agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print()

=== Turn 1: Fetch logs for auth-service and give a one-line summary. ===
  -> model receives 1 tool message(s):
        [auth-service] log-line log-line log-line log ...

=== Turn 2: Now fetch logs for billing-service and summarize in one line. ===
  -> model receives 1 tool message(s):
        [auth-service] log-line log-line log-line log ...
  -> model receives 2 tool message(s):
        [old logs cleared]
        [billing-service] log-line log-line log-line  ...

=== Turn 3: Finally fetch logs for search-service and summarize in one line. ===
  -> model receives 2 tool message(s):
        [old logs cleared]
        [billing-service] log-line log-line log-line  ...
  -> model receives 3 tool message(s):
        [old logs cleared]
        [old logs cleared]
        [search-service] log-line log-line log-line l ...



## Summary

- `ContextEditingMiddleware` automatically trims context **sent to the model** when it grows too large (your saved history is left intact).
- `ClearToolUsesEdit` clears **old tool outputs**: `trigger` (token threshold), `keep` (how many recent results to retain), `placeholder` (replacement text), and optionally `clear_tool_inputs`.
- Use it for long agent sessions to control cost and stay within the context window - no manual message bookkeeping required.